# Data Audit Notebook

Sanity checks for SQLite ingestion outputs with a focus on small-table previews and key field quality checks.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path('../data/interim/books.db')
assert DB_PATH.exists(), f'Missing database at {DB_PATH.resolve()}'
conn = sqlite3.connect(DB_PATH)
print('Connected to:', DB_PATH.resolve())


Connected to: /Users/zacurbiztondo/literary-analysis/data/interim/books.db


In [2]:
# 1) List tables
tables = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    """,
    conn
)
tables

,table_name
0,gemini_content_summaries
1,hardcover_authors
2,hardcover_enrichment
3,nyt_entries
4,openlibrary_enrichment


In [4]:
pd.read_sql_query(
    """
    SELECT *
    FROM gemini_content_summaries
    LIMIT 5
    """,
    conn
)

,isbn13,summary,content_tags_seed,raw_response,last_error,last_checked_at
0,9780060959470,"bell hooks's seminal work, ""All About Love: Ne...",love as practice; self-love; relationships; fe...,"{\n ""isbn13"": ""9780060959470"",\n ""summary"": ...",None,2026-04-19T23:50:40+00:00
1,9780062300553,"J.D. Vance's ""Hillbilly Elegy"" offers a raw an...",Appalachian culture; White working class; Pove...,"{\n ""isbn13"": ""9780062300553"",\n ""summary"": ...",None,2026-04-19T23:50:40+00:00
2,9780062406651,"Mitch Albom's ""The Little Liar"" delves into th...",Deception; Family Relationships; Regret; Redem...,"{\n ""isbn13"": ""9780062406651"",\n ""summary"": ...",None,2026-04-19T23:50:40+00:00
3,9780062424105,"In Julia Quinn's ""Romancing Mister Bridgerton,...",Regency Romance; Friendship to Love; Secret Id...,"{\n ""isbn13"": ""9780062424105"",\n ""summary"": ...",None,2026-04-19T23:50:40+00:00
4,9780062457714,"Mark Manson's ""The Subtle Art of Not Giving a ...",self-help; philosophy; psychology; personal gr...,"{\n ""isbn13"": ""9780062457714"",\n ""summary"": ...",None,2026-04-19T23:50:40+00:00


In [4]:
pd.read_sql_query(
    """
    SELECT COUNT(*)
    FROM openlibrary_enrichment
    WHERE description IS NULL
    """,
    conn
)

,COUNT(*)
0,442


In [5]:
pd.read_sql_query(
    """
    SELECT COUNT(*)
    FROM openlibrary_enrichment
    WHERE work_key IS NOT NULL
    """,
    conn
)

,COUNT(*)
0,595


In [13]:
pd.read_sql_query(
    """
    SELECT COUNT(DISTINCT isbn13)
    FROM nyt_entries
    WHERE list_name IN ('Advice How-To and Miscellaneous', 'Young Adult Paperback')
    """,
    conn
)

,COUNT(DISTINCT isbn13)
0,30


In [14]:
pd.read_sql_query(
    """
    SELECT COUNT(isbn13)
    FROM openlibrary_enrichment
    WHERE isbn13 IN (
        SELECT DISTINCT isbn13
        FROM nyt_entries
        WHERE list_name IN ('Advice How-To and Miscellaneous', 'Young Adult Paperback')
    );
    """,
    conn
)

,COUNT(isbn13)
0,30


In [20]:
pd.read_sql_query(
    """
    SELECT *
    FROM hardcover_enrichment
    WHERE isbn13 IN (
        SELECT DISTINCT isbn13
        FROM nyt_entries
        WHERE list_name IN ('Advice How-To and Miscellaneous', 'Young Adult Paperback')
    ) AND description IS NULL
    ;
    """,
    conn
)

,isbn13,book_id,title,description,rating,ratings_count,users_read_count,cached_tags,last_error,last_checked_at
0,9780593374191,NaN,NaN,None,NaN,NaN,NaN,NaN,edition_not_found,2026-03-20T19:39:13+00:00
1,9780593433317,NaN,NaN,None,NaN,NaN,NaN,NaN,edition_not_found,2026-03-20T19:39:13+00:00
2,9781481438261,NaN,NaN,None,NaN,NaN,NaN,NaN,edition_not_found,2026-03-20T19:54:01+00:00
3,9781728210292,503818.0,The Summer of Broken Rules,None,3.565789,76.0,89.0,"{""Content Warning"": [], ""Genre"": [{""category"":...",NaN,2026-03-20T19:54:01+00:00
4,9798217024636,NaN,NaN,None,NaN,NaN,NaN,NaN,edition_not_found,2026-03-20T19:55:28+00:00


In [9]:
pd.read_sql_query(
    """
    SELECT COUNT(DISTINCT isbn13)
    FROM nyt_entries
    WHERE description IS ""
    """,
    conn
)

,COUNT(DISTINCT isbn13)
0,141


In [12]:
pd.read_sql_query(
    """
    SELECT *
    FROM hardcover_enrichment
    WHERE isbn13 IN (
        "9780143127741",
        "9780593593950",
        "9780399562761",
        "9781595911346",
        "9780399562761",
        "9780399562761",
        "9781594204128"
    )
    """,
    conn
)

,isbn13,book_id,title,description,rating,ratings_count,users_read_count,cached_tags,last_error,last_checked_at
0,9780143127741,358355,"The Body Keeps the Score: Brain, Mind, and Bod...",A pioneering researcher and one of the world’s...,4.329480,346,553,"{""Content Warning"": [{""category"": ""Content War...",None,2026-03-20T19:39:13+00:00
1,9780399562761,705384,The Singularity Is Nearer: When We Merge with AI,The noted inventor and futurist's successor to...,3.857143,14,28,"{""Content Warning"": [], ""Genre"": [{""category"":...",None,2026-03-20T19:39:13+00:00
2,9780593593950,869178,The Coming Wave,An urgent warning of the unprecedented risks t...,3.590909,44,63,"{""Content Warning"": [], ""Genre"": [{""category"":...",None,2026-03-20T19:46:35+00:00
3,9781594204128,1408098,On the Edge: The Art of Risking Everything,"NAMED A MOST-ANTICIPATED BOOK OF 2024 BY FT, T...",3.406250,16,29,"{""Content Warning"": [], ""Genre"": [], ""Mood"": [...",None,2026-03-20T19:54:01+00:00
4,9781595911346,1672504,Love & Whiskey,"""Embark on a captivating journey with Love & W...",NaN,0,0,{},None,2026-03-20T19:54:01+00:00


In [7]:
pd.read_sql_query(
    """
    SELECT *
    FROM openlibrary_enrichment
    LIMIT 10
    """,
    conn
)

,isbn13,work_key,subjects,subject_places,description,last_error,last_checked_at
0,9780062406651,/works/OL35709584W,nyt:combined-print-and-e-book-fiction=2023-12-...,,NaN,None,2026-03-11T01:02:38+00:00
1,9780062457714,/works/OL17590212W,Self-realization|Conduct of life|Conducta de v...,Dallas (Texas)|St. Petersburg (Russia)|Austin ...,"In this book, blogger and former internet entr...",None,2026-03-11T01:02:38+00:00
2,9780062859044,/works/OL26443084W,nyt:combined-print-and-e-book-fiction=2022-09-...,,A small town hides a big secret…\r\n\r\nWho ki...,None,2026-03-11T01:02:38+00:00
3,9780062962843,/works/OL35771233W,nyt:advice-how-to-and-miscellaneous=2023-11-12...,,NaN,None,2026-03-11T01:02:38+00:00
4,9780062976581,/works/OL21208765W,"Literature|Comics & graphic novels, literary|n...",,NaN,None,2026-03-11T01:02:38+00:00
5,9780063076105,/works/OL25713537W,nyt:combined-print-and-e-book-nonfiction=2021-...,,NaN,None,2026-03-11T01:02:38+00:00
6,9780143125471,/works/OL17038299W,University of Washington|Olympic Games (11th :...,United States|Estados Unidos,Daniel James Brown’s robust book tells the sto...,None,2026-03-11T01:02:38+00:00
7,9780143127741,/works/OL18147687W,Post-traumatic stress disorder|Treatment|Thera...,,Trauma is a fact of life. Veterans and their f...,None,2026-03-11T01:02:38+00:00
8,9780307742483,/works/OL17778665W,Case studies|United States. Federal Bureau of ...,Oklahoma|Osage County|Osage County (Okla.)|Uni...,"In the 1920s, the richest people per capita in...",None,2026-03-11T01:02:38+00:00
9,9780316402484,/works/OL35173976W,"Fiction, thrillers, psychological|Fiction, thr...",,NaN,None,2026-03-11T01:02:38+00:00


In [ ]:
nyt_ol = pd.read_csv('../data/processed/nyt_openlibrary_joined.csv')

np.int64(2069)

In [8]:

nyt_ol['subjects'].isna().sum()

np.int64(2069)

In [ ]:
len(nyt_ol)

3150

In [16]:
nyt_ol[nyt_ol['title'] == "THE GIRL WHO FELL BENEATH THE SEA"]

,id,list_name,published_date,rank,weeks_on_list,title,author,publisher,isbn13,isbn10,description,created_at,work_key,subjects,subject_places,ol_description,last_error
1675,1676,Young Adult Paperback,2025-07-06,6,0,THE GIRL WHO FELL BENEATH THE SEA,Axie Oh,Square Fish,9781250866097,NaN,NaN,2026-03-02 00:49:02,/works/OL25219259W,nyt:young-adult-hardcover=2022-03-13|New York ...,NaN,Deadly storms have ravaged Mina’s homeland for...,NaN


In [ ]:
# 2) Row counts per table
counts = []
for t in tables['table_name']:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    counts.append({'table_name': t, 'row_count': n})
counts_df = pd.DataFrame(counts).sort_values('table_name').reset_index(drop=True)
counts_df

In [23]:
# 3) ISBN13 and work_key format checks in openlibrary_enrichment
quality_sql = """
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN isbn13 GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]' THEN 1 ELSE 0 END) AS isbn13_valid_rows,
  SUM(CASE WHEN isbn13 NOT GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]' THEN 1 ELSE 0 END) AS isbn13_invalid_rows,
  SUM(CASE WHEN work_key IS NULL OR TRIM(work_key) = '' THEN 1 ELSE 0 END) AS work_key_blank_rows,
  SUM(CASE WHEN work_key IS NOT NULL AND TRIM(work_key) <> '' AND work_key GLOB '/works/*' THEN 1 ELSE 0 END) AS work_key_valid_rows,
  SUM(CASE WHEN work_key IS NOT NULL AND TRIM(work_key) <> '' AND work_key NOT GLOB '/works/*' THEN 1 ELSE 0 END) AS work_key_invalid_rows
FROM openlibrary_enrichment
"""
pd.read_sql_query(quality_sql, conn)

,total_rows,isbn13_valid_rows,isbn13_invalid_rows,work_key_blank_rows,work_key_valid_rows,work_key_invalid_rows
0,856,848,8,288,568,0


In [28]:
# 4) Show malformed isbn13/work_key rows for inspection
bad_rows_sql = """
SELECT isbn13, work_key, last_error, last_checked_at
FROM openlibrary_enrichment
WHERE isbn13 NOT GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]'
   OR (work_key IS NOT NULL AND TRIM(work_key) <> '' AND work_key NOT GLOB '/works/*')
ORDER BY isbn13
"""
pd.read_sql_query(bad_rows_sql, conn)

,isbn13,work_key,last_error,last_checked_at
0,A00B01K4VZYGY,None,edition_not_found,2026-03-02T01:03:45+00:00
1,DBKACX0234484,None,edition_not_found,2026-03-02T01:03:45+00:00
2,DBKACX0251720,None,edition_not_found,2026-03-02T01:03:45+00:00
3,DBKACX0257961,None,edition_not_found,2026-03-02T01:03:45+00:00
4,DBKACX0286289,None,edition_not_found,2026-03-02T01:03:45+00:00
5,DBKADBL022168,None,edition_not_found,2026-03-02T01:03:45+00:00
6,DBKADBL059878,None,edition_not_found,2026-03-02T01:03:45+00:00
7,DBKADBL063268,None,edition_not_found,2026-03-02T01:03:45+00:00


In [29]:
# 5) Preview small tables in full, otherwise head(20)
def preview_table(table_name: str, small_threshold: int = 100):
    n = conn.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
    if n <= small_threshold:
        q = f'SELECT * FROM {table_name}'
    else:
        q = f'SELECT * FROM {table_name} LIMIT 20'
    df = pd.read_sql_query(q, conn)
    print(f'\n=== {table_name} (rows={n}) ===')
    return df

for t in tables['table_name']:
    display(preview_table(t))


=== nyt_entries (rows=3150) ===


,id,list_name,published_date,rank,weeks_on_list,title,author,publisher,isbn13,isbn10,description,created_at
0,1,Combined Print & E-Book Fiction,2024-12-29,1,8,JAMES,Percival Everett,Doubleday,9780385550369,,A reimagining of “Adventures of Huckleberry Fi...,2026-03-02 00:46:04
1,2,Combined Print & E-Book Fiction,2024-12-29,2,2,WIND AND TRUTH,Brandon Sanderson,Tor,9781250319180,,The fifth book in the Stormlight Archive serie...,2026-03-02 00:46:04
2,3,Combined Print & E-Book Fiction,2024-12-29,3,5,WICKED,Gregory Maguire,Morrow,9780062852847,,A misunderstood girl named Elphaba is declared...,2026-03-02 00:46:04
3,4,Combined Print & E-Book Fiction,2024-12-29,4,45,THE WOMEN,Kristin Hannah,St. Martin's,9781250178633,,A nurse follows her brother to serve during th...,2026-03-02 00:46:04
4,5,Combined Print & E-Book Fiction,2024-12-29,5,73,FOURTH WING,Rebecca Yarros,Red Tower,9781649377371,,Violet Sorrengail is urged by the commanding g...,2026-03-02 00:46:04
5,6,Combined Print & E-Book Fiction,2024-12-29,6,3,THE HOUSE OF CROSS,James Patterson,"Little, Brown",9780316402682,,The 33rd book in the Alex Cross series. Three ...,2026-03-02 00:46:04
6,7,Combined Print & E-Book Fiction,2024-12-29,7,7,THE GOD OF THE WOODS,Liz Moore,Riverhead,9780593418918,,When a 13-year-old girl disappears from an Adi...,2026-03-02 00:46:04
7,8,Combined Print & E-Book Fiction,2024-12-29,8,5,THE WEDDING PEOPLE,Alison Espach,Holt,9781250899569,,A woman who is down on her luck forms an unexp...,2026-03-02 00:46:04
8,9,Combined Print & E-Book Fiction,2024-12-29,9,4,THE FROZEN RIVER,Ariel Lawhon,Vintage,9780593312070,,"In Maine, 1789, a midwife seeks to uncover the...",2026-03-02 00:46:04
9,10,Combined Print & E-Book Fiction,2024-12-29,10,36,A COURT OF THORNS AND ROSES,Sarah J. Maas,Bloomsbury,9781635575569,,"After killing a wolf in the woods, Feyre is ta...",2026-03-02 00:46:04



=== openlibrary_enrichment (rows=856) ===


,isbn13,work_key,subjects,subject_places,description,last_error,last_checked_at
0,9780008725716,NaN,,,NaN,edition_not_found,2026-03-02T00:54:47+00:00
1,9780008728090,/works/OL42395220W,Romance|Christmas|Fiction|Holiday|Contemporary...,,**From the international bestselling author of...,NaN,2026-03-02T00:54:47+00:00
2,9780061968358,NaN,,,NaN,edition_not_found,2026-03-02T00:54:47+00:00
3,9780062200600,/works/OL42552307W,Horror|fiction|time jump,,"Arthur Oakes is a reader, a dreamer, and a stu...",NaN,2026-03-02T00:54:47+00:00
4,9780062300553,/works/OL17357665W,Mountain people|Family|Social mobility|Working...,United States|Appalachian Region|Kentucky,From a former marine and Yale Law School gradu...,NaN,2026-03-02T00:54:47+00:00
5,9780062406682,/works/OL44220771W,,,"When he is eight years old, Alfie Logan discov...",NaN,2026-03-02T00:54:47+00:00
6,9780062407801,/works/OL18819818W,Negotiation|Negotiation in business|Business c...,,"""A negotiation guide from a former FBI Hostage...",NaN,2026-03-02T00:54:47+00:00
7,9780062477521,NaN,,,NaN,edition_not_found,2026-03-02T00:54:47+00:00
8,9780062852847,/works/OL34638280W,,,NaN,NaN,2026-03-02T00:54:47+00:00
9,9780062863102,/works/OL43273995W,,,The extraordinary life of Cher can be told by ...,NaN,2026-03-02T00:54:47+00:00


In [5]:
# 6) Optional: close connection when done
conn.close()